In [16]:
import torch
import torch.nn.functional as F
import torch_geometric.nn as geom
from torch_geometric.nn import GraphConv, GCNConv, GATConv, GATv2Conv, global_mean_pool, BatchNorm
import torch.nn as nn
from torch.nn import Linear, Dropout, LeakyReLU
import inspect

In [30]:
def get_remaining_params(args, cls):
    signature = inspect.signature(cls.__init__)
    
    parameters = [
        param.name for param in signature.parameters.values() 
        if param.name != 'self'
    ]

    return {key:args[key] for key in set(args.keys()).intersection(parameters)}

In [11]:
layer_args = {"in_channels": 77,
              "out_channels": 128,
              "heads": 8,
              "edge_dim": 0,
              "concat": True,}
a = GATv2Conv(**layer_args)

In [12]:
gnn_class = GATConv

b = gnn_class(**layer_args)

In [48]:
gnn_class = GCNConv
c = GCNConv(**get_remaining_params(layer_args, gnn_class))

In [32]:
get_remaining_params(layer_args, GraphConv)

{'in_channels': 77, 'out_channels': 128}

In [41]:
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from preprocessing import get_loaders
from pathlib import Path


target = "donor"
data="../Data/amd"
print(f"Training {target}")

data_dirs = {}
for data_type in ['TER', 'VEGF', 'Both', 'donor']:
    data_dirs[f"Train_{data_type}"] = f"{data}/{data_type}/train_singular_donors.pkl"
    data_dirs[f"Valid_{data_type}"] = f"{data}/{data_type}/valid_singular_donors.pkl"
    data_dirs[f"Test_{data_type}"] = f"{data}/{data_type}/test_singular_donors.pkl"

train_loader, val_loader, test_loader, data_details = get_loaders(data_dirs, target, 16)
train_loaders = [train_loader]
val_loaders = [val_loader]
test_loaders = [test_loader]

Training donor


/opt/homebrew/Caskroom/miniforge/base/envs/qbam_gnn/lib/python3.12/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(io.Byt

###################################
Number of training graphs: 3296
Number of validation graphs: 412
Number of test graphs: 412
Number of features: 77
Number of targets: 1
###################################


In [42]:
batch = next(iter(train_loaders[0]))

In [43]:
batch

DataBatch(x=[5370, 77], edge_index=[2, 127218], edge_weights=[127218], y=[16], batch=[5370], ptr=[17])

In [47]:
a(batch.x, batch.edge_index).shape

torch.Size([5370, 1024])

In [46]:
b(batch.x, batch.edge_index).shape

torch.Size([5370, 1024])

In [49]:
c(batch.x, batch.edge_index).shape

torch.Size([5370, 128])